In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}, {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")

CUDA available: True
GPU count: 2
GPU 0: Tesla T4, 15.6 GB
GPU 1: Tesla T4, 15.6 GB


In [2]:
!pip install -q -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 77.3 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 45.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 28.9 MB/s eta 0:00:00


In [3]:
import os, json, time, gc, re
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BNB_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

# Load HF token if present (needed for official gated repos like meta-llama/*)
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    print("HF_TOKEN loaded.")
except Exception:
    hf_token = None
    print("No HF_TOKEN secret found — official gated repos will fail over to mirrors automatically.")

print("Setup ready. Checkpoints saved to:", CHECKPOINT_DIR)

HF_TOKEN loaded.
Setup ready. Checkpoints saved to: /kaggle/working/checkpoints


In [4]:
def load_model_with_fallback(official_id, fallback_id, token=None):
    """
    Tries the official (possibly gated) repo first. If access is denied
    (GatedRepoError / 403), falls back to the ungated mirror automatically.
    Returns (tokenizer, model, which_id_used).
    """
    try:
        print(f"Trying official repo: {official_id}")
        tok = AutoTokenizer.from_pretrained(official_id, token=token)
        mdl = AutoModelForCausalLM.from_pretrained(
            official_id, quantization_config=BNB_CONFIG, device_map="auto", token=token
        )
        print(f"Loaded OFFICIAL repo: {official_id}")
        return tok, mdl, official_id
    except Exception as e:
        print(f"Official repo failed ({type(e).__name__}) — falling back to mirror: {fallback_id}")
        tok = AutoTokenizer.from_pretrained(fallback_id)
        mdl = AutoModelForCausalLM.from_pretrained(fallback_id, quantization_config=BNB_CONFIG, device_map="auto")
        print(f"Loaded FALLBACK repo: {fallback_id}")
        return tok, mdl, fallback_id

print("Model loader with fallback ready.")

Model loader with fallback ready.


In [5]:
PAYOFFS = {
    ("C", "C"): (3, 3),
    ("C", "D"): (0, 5),
    ("D", "C"): (5, 0),
    ("D", "D"): (1, 1),
}

def tit_for_tat_opponent(history):
    if len(history) == 0:
        return "C"
    return history[-1]["your_action"]

def build_prompt(history, current_round, total_rounds):
    game_rules = (
        "You are playing a repeated Prisoner's Dilemma game with an opponent for "
        f"{total_rounds} rounds. Each round, you and your opponent simultaneously choose "
        "either 'C' (cooperate) or 'D' (defect).\n\n"
        "Payoffs per round:\n"
        "- Both cooperate: you get 3, opponent gets 3\n"
        "- You cooperate, opponent defects: you get 0, opponent gets 5\n"
        "- You defect, opponent cooperates: you get 5, opponent gets 0\n"
        "- Both defect: you get 1, opponent gets 1\n\n"
        "Your goal is to maximize your own total payoff across all rounds.\n\n"
    )
    if len(history) == 0:
        history_text = "This is round 1. No history yet.\n"
    else:
        history_text = "History so far:\n"
        for i, h in enumerate(history):
            history_text += (
                f"Round {i+1}: You played {h['your_action']}, "
                f"opponent played {h['opponent_action']}, you scored {h['your_payoff']}.\n"
            )
    instruction = (
        f"\nIt is now round {current_round} of {total_rounds}. "
        "Think briefly about your strategy, then respond with your final decision "
        "on a new line in exactly this format:\nDECISION: C\nor\nDECISION: D"
    )
    return game_rules + history_text + instruction

def parse_action(response_text):
    match = re.search(r"DECISION:\s*([CD])", response_text.upper())
    if match:
        return match.group(1)
    if "D" in response_text[-20:].upper():
        return "D"
    return "C"

print("Game logic loaded.")

Game logic loaded.


In [6]:
def play_game_checkpointed(model, tokenizer, model_name, game_name="prisoners_dilemma",
                             total_rounds=10, max_new_tokens=250, verbose=True):
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"{model_name}_{game_name}.json")

    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, "r") as f:
            history = json.load(f)
        print(f"Resuming '{model_name}' on '{game_name}' — {len(history)}/{total_rounds} rounds already done.")
    else:
        history = []
        print(f"Starting fresh: '{model_name}' on '{game_name}'.")

    for round_num in range(len(history) + 1, total_rounds + 1):
        prompt_text = build_prompt(history, round_num, total_rounds)
        messages = [
            {"role": "system", "content": "You are a strategic game-playing agent."},
            {"role": "user", "content": prompt_text}
        ]
        inputs = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
        ).to(model.device)

        t0 = time.time()
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        elapsed = time.time() - t0

        response_text = tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
        your_action = parse_action(response_text)
        opponent_action = tit_for_tat_opponent(history)
        your_payoff, opponent_payoff = PAYOFFS[(your_action, opponent_action)]

        history.append({
            "round": round_num, "your_action": your_action, "opponent_action": opponent_action,
            "your_payoff": your_payoff, "opponent_payoff": opponent_payoff, "full_response": response_text,
        })

        with open(checkpoint_path, "w") as f:
            json.dump(history, f, indent=2)

        rounds_left = total_rounds - round_num
        eta_min = (rounds_left * elapsed) / 60
        if verbose:
            print(f"[{model_name}] Round {round_num}/{total_rounds} | Action: {your_action} | "
                  f"Opp: {opponent_action} | Payoff: {your_payoff} | {elapsed:.0f}s/round | ETA: {eta_min:.1f} min")

    return history

print("Checkpointed game function ready.")

Checkpointed game function ready.


In [7]:
def compute_regret(game_history, payoff_matrix, opponent_history):
    actions = ["C", "D"]
    total_rounds = len(game_history)
    actual_payoff = sum(h["your_payoff"] for h in game_history)
    best_fixed_payoff = -np.inf
    for fixed_action in actions:
        total = sum(payoff_matrix[(fixed_action, opp)][0] for opp in opponent_history)
        best_fixed_payoff = max(best_fixed_payoff, total)
    regret = best_fixed_payoff - actual_payoff
    return {
        "actual_payoff": actual_payoff,
        "best_fixed_payoff": best_fixed_payoff,
        "regret_vs_best_fixed": regret,
        "avg_regret_per_round": regret / total_rounds,
    }

def simulate_ftrl_live(opponent_strategy_fn, payoff_matrix, total_rounds, learning_rate=0.5, seed=42):
    rng = np.random.default_rng(seed)
    actions = ["C", "D"]
    cumulative_scores = {"C": 0.0, "D": 0.0}
    history, total_payoff = [], 0
    for _ in range(total_rounds):
        scores = np.array([cumulative_scores[a] for a in actions])
        probs = np.exp(learning_rate * scores) / np.exp(learning_rate * scores).sum()
        chosen = rng.choice(actions, p=probs)
        opp_act = opponent_strategy_fn(history)
        payoff = payoff_matrix[(chosen, opp_act)][0]
        total_payoff += payoff
        history.append({"your_action": chosen, "opponent_action": opp_act})
        for a in actions:
            cumulative_scores[a] += payoff_matrix[(a, opp_act)][0]
    return {"total_payoff": total_payoff, "actions": [h["your_action"] for h in history]}

def simulate_ftpl_live(opponent_strategy_fn, payoff_matrix, total_rounds, noise_scale=1.0, seed=42):
    rng = np.random.default_rng(seed)
    actions = ["C", "D"]
    cumulative_scores = {"C": 0.0, "D": 0.0}
    history, total_payoff = [], 0
    for _ in range(total_rounds):
        noisy_scores = {a: cumulative_scores[a] + rng.normal(0, noise_scale) for a in actions}
        chosen = max(noisy_scores, key=noisy_scores.get)
        opp_act = opponent_strategy_fn(history)
        payoff = payoff_matrix[(chosen, opp_act)][0]
        total_payoff += payoff
        history.append({"your_action": chosen, "opponent_action": opp_act})
        for a in actions:
            cumulative_scores[a] += payoff_matrix[(a, opp_act)][0]
    return {"total_payoff": total_payoff, "actions": [h["your_action"] for h in history]}

def run_full_evaluation(model, tokenizer, model_name, game_name="prisoners_dilemma",
                          total_rounds=10, max_new_tokens=250, opponent_fn=tit_for_tat_opponent, payoff_matrix=PAYOFFS):
    game_history = play_game_checkpointed(model, tokenizer, model_name, game_name, total_rounds, max_new_tokens, verbose=True)
    opponent_actions = [h["opponent_action"] for h in game_history]
    regret_result = compute_regret(game_history, payoff_matrix, opponent_actions)
    ftrl_result = simulate_ftrl_live(opponent_fn, payoff_matrix, total_rounds)
    ftpl_result = simulate_ftpl_live(opponent_fn, payoff_matrix, total_rounds)
    return {
        "model": model_name, "game": game_name,
        "actual_payoff": regret_result["actual_payoff"],
        "best_fixed_payoff": regret_result["best_fixed_payoff"],
        "regret_vs_best_fixed": regret_result["regret_vs_best_fixed"],
        "ftrl_payoff": ftrl_result["total_payoff"], "ftpl_payoff": ftpl_result["total_payoff"],
        "regret_vs_ftrl": ftrl_result["total_payoff"] - regret_result["actual_payoff"],
        "regret_vs_ftpl": ftpl_result["total_payoff"] - regret_result["actual_payoff"],
        "beats_ftrl": regret_result["actual_payoff"] >= ftrl_result["total_payoff"],
        "beats_ftpl": regret_result["actual_payoff"] >= ftpl_result["total_payoff"],
        "action_sequence": [h["your_action"] for h in game_history],
        "raw_history": game_history,
    }

print("Regret/FTRL/FTPL calculators ready.")

Regret/FTRL/FTPL calculators ready.


In [ ]:
LLAMA_OFFICIAL = "meta-llama/Meta-Llama-3-8B-Instruct"
LLAMA_FALLBACK = "NousResearch/Meta-Llama-3-8B-Instruct"

if "model_llama" not in globals():
    tokenizer_llama, model_llama, llama_id_used = load_model_with_fallback(
        LLAMA_OFFICIAL, LLAMA_FALLBACK, token=os.environ.get("HF_TOKEN")
    )

result_llama = run_full_evaluation(model_llama, tokenizer_llama, model_name="Llama-3-8B-Instruct", max_new_tokens=250)
print(result_llama["model"], "| Actual:", result_llama["actual_payoff"],
      "| FTRL:", result_llama["ftrl_payoff"], "| FTPL:", result_llama["ftpl_payoff"])

In [ ]:
def sample_action_counts_checkpointed(model, tokenizer, model_name, context_label, history_for_context,
                                        current_round, total_rounds, n_samples=15, temperature=0.7,
                                        max_new_tokens=250):
    """
    Runs n_samples independent SAMPLED generations at a FIXED history/context,
    and counts how many times the model chose C vs D. This is what actually lets
    tau be estimated — matches Jia et al. 2025's frequency-based method, adapted
    to our repeated-game contexts (paper used 30 one-shot trials on a static
    matrix; we use fewer samples across a few representative round-states due to
    compute limits on T4 — a scope limitation to state honestly in any write-up).
    """
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"{model_name}_{context_label}_samples.json")

    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, "r") as f:
            samples = json.load(f)
        print(f"Resuming sampling '{context_label}' — {len(samples)}/{n_samples} samples already done.")
    else:
        samples = []
        print(f"Starting fresh sampling: '{context_label}'.")

    prompt_text = build_prompt(history_for_context, current_round, total_rounds)
    messages = [
        {"role": "system", "content": "You are a strategic game-playing agent."},
        {"role": "user", "content": prompt_text}
    ]

    for i in range(len(samples), n_samples):
        inputs = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
        ).to(model.device)

        output = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=True, temperature=temperature, top_p=0.9
        )
        response_text = tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
        action = parse_action(response_text)
        samples.append(action)

        with open(checkpoint_path, "w") as f:
            json.dump(samples, f, indent=2)

        print(f"  [{model_name}/{context_label}] sample {i+1}/{n_samples}: {action}")

    counts = {"C": samples.count("C"), "D": samples.count("D")}
    opponent_action_for_context = tit_for_tat_opponent(history_for_context)
    return {"context_label": context_label, "counts": counts, "opponent_action": opponent_action_for_context}

# --- Define 3 representative game-states from the ALREADY-COMPLETED greedy game ---
# early = no history (round 1), mid = after round 5, late = after round 9
llama_contexts = [
    ("early", [], 1, 10),
    ("mid", result_llama["raw_history"][:5], 6, 10),
    ("late", result_llama["raw_history"][:9], 10, 10),
]

llama_sample_results = []
for label, hist, round_num, total in llama_contexts:
    res = sample_action_counts_checkpointed(model_llama, tokenizer_llama, "Llama-3-8B-Instruct",
                                              label, hist, round_num, total, n_samples=15, temperature=0.7)
    llama_sample_results.append(res)

print("\nLlama-3-8B sampling summary:", llama_sample_results)

In [ ]:
if "model_llama" in globals():
    del model_llama, tokenizer_llama
    gc.collect()
    torch.cuda.empty_cache()

R1_OFFICIAL = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"
R1_FALLBACK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"  # same repo; no separate mirror needed, not gated

if "model_r1" not in globals():
    tokenizer_r1, model_r1, r1_id_used = load_model_with_fallback(
        R1_OFFICIAL, R1_FALLBACK, token=os.environ.get("HF_TOKEN")
    )

# max_new_tokens cited: Jia et al. 2025 (arXiv:2502.20432) Table 8, mixed-motive/PD-category — mean 2318, max 3742
result_r1 = run_full_evaluation(model_r1, tokenizer_r1, model_name="DeepSeek-R1-Distill-Qwen-14B", max_new_tokens=4000)
print(result_r1["model"], "| Actual:", result_r1["actual_payoff"],
      "| FTRL:", result_r1["ftrl_payoff"], "| FTPL:", result_r1["ftpl_payoff"])

In [ ]:
r1_contexts = [
    ("early", [], 1, 10),
    ("mid", result_r1["raw_history"][:5], 6, 10),
    ("late", result_r1["raw_history"][:9], 10, 10),
]

r1_sample_results = []
for label, hist, round_num, total in r1_contexts:
    res = sample_action_counts_checkpointed(model_r1, tokenizer_r1, "DeepSeek-R1-Distill-Qwen-14B",
                                              label, hist, round_num, total, n_samples=15,
                                              temperature=0.7, max_new_tokens=4000)
    r1_sample_results.append(res)

print("\nDeepSeek-R1 sampling summary:", r1_sample_results)

In [ ]:
import math
from scipy.optimize import minimize_scalar

def fit_tau_from_counts(context_results, payoff_matrix, gamma=1.0, max_k=10):
    """
    context_results: list of dicts like {"counts": {"C": 12, "D": 3}, "opponent_action": "C"}
    Fits a single pooled tau using weighted log-likelihood across all contexts —
    matches the paper's use of action-count frequencies (their cij) rather than
    single deterministic picks.
    """
    actions = ["C", "D"]

    def poisson_weight(k, tau):
        return (tau**k) * np.exp(-tau) / math.factorial(k)

    def level_k_strategy(k, opp_act, prev_levels):
        if k == 0:
            return {"C": 0.5, "D": 0.5}
        expected_utils = {}
        for a in actions:
            expected_utils[a] = payoff_matrix[(a, opp_act)][0]
        max_u = max(expected_utils.values())
        exp_vals = {a: np.exp(gamma * k * (expected_utils[a] - max_u)) for a in actions}
        total = sum(exp_vals.values())
        return {a: exp_vals[a] / total for a in actions}

    def neg_log_likelihood(tau):
        total_ll = 0
        for ctx in context_results:
            opp_act = ctx["opponent_action"]
            levels = {0: {"C": 0.5, "D": 0.5}}
            for k in range(1, max_k):
                levels[k] = level_k_strategy(k, opp_act, levels)
            for action in actions:
                count = ctx["counts"].get(action, 0)
                if count == 0:
                    continue
                p_action = sum(poisson_weight(k, tau) * levels[k][action] for k in range(max_k))
                p_action = max(p_action, 1e-10)
                total_ll += count * np.log(p_action)
        return -total_ll

    result = minimize_scalar(neg_log_likelihood, bounds=(0.01, 8.0), method='bounded')
    return result.x

tau_llama = fit_tau_from_counts(llama_sample_results, PAYOFFS)
tau_r1 = fit_tau_from_counts(r1_sample_results, PAYOFFS)

print(f"Llama-3-8B  | Regret vs FTRL: {result_llama['regret_vs_ftrl']:+d} | τ (from {sum(sum(c['counts'].values()) for c in llama_sample_results)} samples): {tau_llama:.2f}")
print(f"DeepSeek-R1 | Regret vs FTRL: {result_r1['regret_vs_ftrl']:+d} | τ (from {sum(sum(c['counts'].values()) for c in r1_sample_results)} samples): {tau_r1:.2f}")